# LoanGuard AI - MLOps Lab
## Data & Model Versioning + Registry

---

### Your Mission

You've just joined **LoanGuard AI**, a fintech startup that predicts loan defaults.
Their ML team has a serious problem: models keep breaking in production and nobody can explain why.
Predictions changed last week, accuracy dropped, and a roll back took 3 days.

**Your job:** Fix their MLOps stack - layer by layer - using data versioning, experiment tracking, model versioning, and a model registry.

---

### Lab Structure

| Task | Topic | Real-World Problem You Solve |
|------|-------|------------------------------|
| **Task 1** | Data Versioning with DVC | Reproduce a model that broke 2 months ago |
| **Task 2** | Experiment Tracking with MLflow | Compare 3 model candidates before deploying |
| **Task 3** | Model Versioning | Name and register a model so the team can roll back |
| **Task 4** | Registry Lifecycle | Promote a model through staging → production → archived |

---

### How the Lab Works

- Each task has a ** Beginner path** (fill in the blanks) and a ** Advanced path** (write from scratch)
- Use the ** Hint system** - try first, then reveal hints one at a time
- Run the ** Auto-check** cells to verify your work before moving on
- **You do not need to run DVC or MLflow CLI** - everything is done in Python

---

### Setup - Run this first

In [1]:
# --- SETUP - Run this cell before anything else ----------------------
import os, json, hashlib, shutil, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler
## added line
# Install required packages for the lab
!pip install mlflow scikit-learn pandas numpy

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import pickle, time

warnings.filterwarnings("ignore")

# Lab workspace
LAB_DIR = Path("loanGuard_lab")
LAB_DIR.mkdir(exist_ok=True)
(LAB_DIR / "data").mkdir(exist_ok=True)
(LAB_DIR / "models").mkdir(exist_ok=True)
(LAB_DIR / "dvc_cache").mkdir(exist_ok=True)
MLFLOW_URI = f"sqlite:///{LAB_DIR}/mlflow.db"

print(" Setup complete - LoanGuard AI Lab is ready")
print(f"   Workspace : {LAB_DIR.resolve()}")
print(f"   MLflow DB : {MLFLOW_URI}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 10.8 MB/s eta 0:00:00
 Setup complete - LoanGuard AI Lab is ready
   Workspace : /content/loanGuard_lab
   MLflow DB : sqlite:///loanGuard_lab/mlflo

### Simulated Data Generator

LoanGuard AI receives loan applications with customer financial data.
Run the cell below to simulate **three versions** of their dataset - representing how data changed over time.

>  **Don't modify this cell.** It simulates real-world upstream data changes you'll need to detect and version.

In [2]:
# ----- DATA SIMULATOR - Do not modify ----------------------------------
np.random.seed(42)

def make_loan_dataset(n=2000, schema_version="v1", seed=42):
    """Simulate LoanGuard's loan application data at different points in time."""
    rng = np.random.RandomState(seed)
    age            = rng.randint(22, 65, n)
    income         = rng.normal(55000, 20000, n).clip(15000, 200000)
    loan_amount    = rng.normal(12000, 5000, n).clip(1000, 50000)
    credit_score   = rng.normal(680, 80, n).clip(300, 850)
    employment_yrs = rng.randint(0, 30, n)
    debt_ratio     = rng.uniform(0.05, 0.65, n)

    # Default probability driven by risk factors
    risk = (
        -0.3 * (credit_score - 680) / 80
        + 0.4 * debt_ratio
        + 0.2 * (loan_amount / income)
        - 0.1 * employment_yrs / 10
    )
    default = (1 / (1 + np.exp(-risk * 3 + rng.normal(0, 0.5, n))) > 0.5).astype(int)

    df = pd.DataFrame({
        "age": age, "annual_income": income.round(2),
        "loan_amount": loan_amount.round(2), "credit_score": credit_score.round(1),
        "employment_years": employment_yrs, "debt_to_income": debt_ratio.round(4),
        "default": default
    })

    # v2: upstream adds a new column (common real-world schema drift)
    if schema_version in ("v2", "v3"):
        df["num_accounts"] = rng.randint(1, 10, n)

    # v3: a data quality issue introduces nulls (another common real-world problem)
    if schema_version == "v3":
        null_idx = rng.choice(n, size=int(n * 0.15), replace=False)
        df.loc[null_idx, "credit_score"] = np.nan

    return df

# Generate and save three dataset versions
datasets = {}
for ver, schema, seed in [("v1", "v1", 42), ("v2", "v2", 42), ("v3", "v3", 99)]:
    df = make_loan_dataset(schema_version=schema, seed=seed)
    path = LAB_DIR / "data" / f"loans_{ver}.csv"
    df.to_csv(path, index=False)
    datasets[ver] = df
    print(f"  loans_{ver}.csv  |  {len(df):,} rows  |  {df.shape[1]} cols  |  "
          f"{df['default'].mean():.1%} default rate  |  nulls: {df.isnull().sum().sum()}")

print("\n Three dataset versions ready")

  loans_v1.csv  |  2,000 rows  |  7 cols  |  58.1% default rate  |  nulls: 0
  loans_v2.csv  |  2,000 rows  |  8 cols  |  58.1% default rate  |  nulls: 0
  loans_v3.csv  |  2,000 rows  |  8 cols  |  54.6% default rate  |  nulls: 300

 Three dataset versions ready


---
# TASK 1 - Data Versioning with DVC
> Can you prove which data trained the broken model?
---

##  Task 1 - Data Versioning with DVC

### Scenario
> LoanGuard's model accuracy dropped from 87% to 71% last month. The engineering team says 'the code didn't change'. You suspect a silent data change. You need to (a) version the dataset snapshots so they can be reproduced later, (b) detect what changed between versions, and (c) implement schema validation to catch future changes automatically.

### Objective
By the end of this task you will be able to:
- Simulate DVC-style versioning using checksums and metadata (`.dvc` pointer files)
- Compare two dataset versions and identify schema/distribution drift
- Write a schema validator that raises an alert before a bad dataset enters the pipeline

---

###  Background - How DVC Versioning Works

DVC doesn't store data in Git. It stores a **tiny pointer file** (`.dvc`) containing:
- A **checksum** (MD5 hash) of the data file
- The file path and size
- The remote storage location

When you run `dvc pull`, DVC uses the checksum to fetch the exact data file.

In this task you'll **simulate this mechanism in pure Python** - the same logic DVC uses internally.

---

###  Choose Your Path

| Path | Description |
|------|-------------|
|  **Beginner** | Scaffolded code - fill in the blanks (`# YOUR CODE HERE`) |
|  **Advanced** | Open-ended - write from scratch using only the requirements |

**You are on the  Beginner path.** Skip to the  Advanced section if you want a challenge.

---

###  1A - Create a DVC-style pointer file

Create a function that takes a CSV file path and produces a `.dvc` metadata dictionary
containing: `md5`, `size`, `path`, `rows`, `cols`, and `created_at`.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `hashlib.md5()` to compute the file checksum. Open the file in binary mode (`'rb'`) and feed it to `.update()`.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

For metadata: `os.path.getsize(path)` gives size in bytes. Load with `pd.read_csv()` and use `.shape` for rows/cols.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def create_dvc_pointer(csv_path):
    path = Path(csv_path)
    with open(path, 'rb') as f:
        md5 = hashlib.md5(f.read()).hexdigest()
    df = pd.read_csv(path)
    return {
        'md5': md5, 'path': str(path),
        'size': os.path.getsize(path),
        'rows': len(df), 'cols': df.shape[1],
        'created_at': datetime.now().isoformat()
    }
```

</details>

In [3]:
#  BEGINNER 1A - Create DVC-style pointer files
# ----------------------------------------------------------------------------------------

def create_dvc_pointer(csv_path):
    """
    Create a DVC-style metadata pointer for a dataset file.

    Parameters
    ----------
    csv_path : str or Path
        Path to the CSV file to version.

    Returns
    -------
    dict  with keys: md5, path, size, rows, cols, created_at
    """


    # Step 1: Compute MD5 checksum of the file
    # YOUR CODE HERE


        # convert to Path object
    path = Path(csv_path)

    # Step 1: compute md5 hash of file
    hash_md5 = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)

    md5 = hash_md5.hexdigest()


    # Step 2: Load the CSV to get row/column counts
    # YOUR CODE HERE
    df = pd.read_csv(path)

    # Step 3: Build and return the metadata dictionary
    return {
        "md5": md5,
        "path": str(path),
        "size": os.path.getsize(path),
        "rows": len(df),
        "cols": df.shape[1],
        "created_at": datetime.now().isoformat()
    }

# ----- Test it -----
pointer_v1 = create_dvc_pointer(LAB_DIR / "data" / "loans_v1.csv")
print("DVC pointer for loans_v1:")
print(json.dumps(pointer_v1, indent=2))

DVC pointer for loans_v1:
{
  "md5": "32749950b434fb1f5e15bc9573a4ccd0",
  "path": "loanGuard_lab/data/loans_v1.csv",
  "size": 76059,
  "rows": 2000,
  "cols": 7,
  "created_at": "2026-03-13T10:33:04.396088"
}


###  1B - Version all three datasets and save pointer files

Call your function on all three loan datasets and save each pointer as a `.dvc` JSON file.

In [4]:
# BEGINNER 1B - Version all three datasets
# --------------------------------------------------

dvc_store = {}   # will hold {version: pointer_dict}

for version in ["v1", "v2", "v3"]:
    csv_path = LAB_DIR / "data" / f"loans_{version}.csv"

    # Step 1: Create pointer
    pointer = create_dvc_pointer(csv_path)

    # Step 2: Save pointer to a .dvc JSON file
    dvc_file = LAB_DIR / "data" / f"loans_{version}.csv.dvc"

    with open(dvc_file, "w") as f:
        json.dump(pointer, f, indent=2)

    dvc_store[version] = pointer

    print(f"  loans_{version}.csv  →  md5: {pointer['md5'][:12]}...  "
          f"({pointer['rows']} rows, {pointer['cols']} cols)")

print("\nAll versions tracked")

  loans_v1.csv  →  md5: 32749950b434...  (2000 rows, 7 cols)
  loans_v2.csv  →  md5: 3b43198058ae...  (2000 rows, 8 cols)
  loans_v3.csv  →  md5: b8ac0fe8ebdd...  (2000 rows, 8 cols)

All versions tracked


###  1C - Compare two versions and detect drift

Write a function that compares two DVC pointers and the underlying CSVs to detect:
- Schema changes (new/removed columns)
- Row count changes
- Statistical drift in numeric columns (mean shift > 5%)
- Data quality issues (null counts)

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Load both CSVs with `pd.read_csv()`. Compare `df1.columns` and `df2.columns` using set operations to find added/removed columns.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

For numeric drift, iterate over shared columns: `abs(df2[col].mean() - df1[col].mean()) / (df1[col].mean() + 1e-9)`. Flag anything above 0.05.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def compare_versions(ptr_a, ptr_b):
    df_a, df_b = pd.read_csv(ptr_a['path']), pd.read_csv(ptr_b['path'])
    cols_a, cols_b = set(df_a.columns), set(df_b.columns)
    report = {
        'added_cols':   list(cols_b - cols_a),
        'removed_cols': list(cols_a - cols_b),
        'row_delta':    ptr_b['rows'] - ptr_a['rows'],
        'null_delta':   int(df_b.isnull().sum().sum() - df_a.isnull().sum().sum()),
        'drift': {}
    }
    for col in cols_a & cols_b:
        if df_a[col].dtype in [float, int]:
            shift = abs(df_b[col].mean() - df_a[col].mean()) / (abs(df_a[col].mean()) + 1e-9)
            if shift > 0.05:
                report['drift'][col] = round(shift, 4)
    return report
```

</details>

In [5]:
#  BEGINNER 1C - Detect drift between dataset versions
# ----------------------------------------------------------

def compare_dataset_versions(ptr_a, ptr_b):
    """
    Compare two versioned datasets and produce a drift report.

    Returns a dict with:
      - added_cols    : columns present in b but not a
      - removed_cols  : columns present in a but not b
      - row_delta     : change in row count
      - null_delta    : change in total null count
      - drift         : dict of {col: relative_mean_shift} for drifted columns
    """
    df_a = pd.read_csv(ptr_a["path"])
    df_b = pd.read_csv(ptr_b["path"])

    cols_a = set(df_a.columns)
    cols_b = set(df_b.columns)

    report = {
        # "added_cols":   # YOUR CODE HERE - columns in b not in a
        # "removed_cols": # YOUR CODE HERE - columns in a not in b
        # "row_delta":    # YOUR CODE HERE - difference in row count
        # "null_delta":   # YOUR CODE HERE - difference in total nulls
        # "drift":        {}
           "added_cols": list(cols_b - cols_a),
          "removed_cols": list(cols_a - cols_b),
          "row_delta": ptr_b["rows"] - ptr_a["rows"],
          "null_delta": int(df_b.isnull().sum().sum() - df_a.isnull().sum().sum()),
          "drift": {}
    }

    # Detect numeric drift in shared columns
    shared_numeric = [
        c for c in cols_a & cols_b
        if df_a[c].dtype in [np.float64, np.int64]
    ]
    for col in shared_numeric:
        mean_a = df_a[col].mean()
        mean_b = df_b[col].mean()
        # YOUR CODE HERE - compute relative shift and add to report if > 5%
        shift = abs(mean_b - mean_a) / (abs(mean_a) + 1e-9)

        if shift > 0.05:
            report["drift"][col] = round(shift, 4)


    return report

# ----- Compare v1 → v2, then v2 → v3 -----
print("═" * 55)
print("Comparing loans_v1  →  loans_v2")
print("═" * 55)
r12 = compare_dataset_versions(dvc_store["v1"], dvc_store["v2"])
print(json.dumps(r12, indent=2))

print("\n" + "═" * 55)
print("Comparing loans_v2  →  loans_v3  (the bad version!)")
print("═" * 55)
r23 = compare_dataset_versions(dvc_store["v2"], dvc_store["v3"])
print(json.dumps(r23, indent=2))

═══════════════════════════════════════════════════════
Comparing loans_v1  →  loans_v2
═══════════════════════════════════════════════════════
{
  "added_cols": [
    "num_accounts"
  ],
  "removed_cols": [],
  "row_delta": 0,
  "null_delta": 0,
  "drift": {}
}

═══════════════════════════════════════════════════════
Comparing loans_v2  →  loans_v3  (the bad version!)
═══════════════════════════════════════════════════════
{
  "added_cols": [],
  "removed_cols": [],
  "row_delta": 0,
  "null_delta": 300,
  "drift": {
    "default": 0.0594
  }
}


###  1D - Schema Validator

Write a schema validator that checks incoming data against a **golden schema** (captured from a known-good dataset).
It must raise a `ValueError` with a descriptive message if validation fails.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Store the golden schema as a dict: `{'columns': list, 'dtypes': dict, 'null_thresholds': dict}`. Compare incoming data against it.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Check three things in order: (1) all expected columns present, (2) null rate per column below threshold, (3) value ranges reasonable.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def build_golden_schema(df, null_tolerance=0.05):
    return {
        'columns': list(df.columns),
        'null_rates': {c: df[c].isnull().mean() for c in df.columns},
        'null_tolerance': null_tolerance,
        'ranges': {c: (df[c].min(), df[c].max())
                   for c in df.select_dtypes(include=np.number).columns}
    }

def validate_schema(df, schema):
    errors = []
    missing = set(schema['columns']) - set(df.columns)
    if missing: errors.append(f'Missing columns: {missing}')
    for col in schema['columns']:
        if col in df.columns:
            null_rate = df[col].isnull().mean()
            if null_rate > schema['null_tolerance']:
                errors.append(f'{col}: null rate {null_rate:.1%} > tolerance')
    if errors:
        raise ValueError('Schema validation failed:\n' + '\n'.join(errors))
    return True
```

</details>

In [6]:
#  BEGINNER 1D - Schema Validator
# ----------------------------------------------------------

def build_golden_schema(df, null_tolerance=0.05):
    """
    Capture the schema of a known-good dataset.

    Returns a dict with: columns, null_rates, null_tolerance, ranges
    """
    # YOUR CODE HERE
    # store structure of dataset so new datasets can be validated later
    return {
        "columns": list(df.columns),

        # storing current null rates to understand what is normal in the clean dataset
        "null_rates": {c: df[c].isnull().mean() for c in df.columns},

        # small tolerance allowed because real datasets may have a few nulls
        "null_tolerance": null_tolerance,

        # storing numeric ranges so we know the normal value limits
        "ranges": {
            c: (df[c].min(), df[c].max())
            for c in df.select_dtypes(include=np.number).columns
        }
    }


def validate_schema(df, schema):
    """
    Validate a new DataFrame against a golden schema.
    Raise ValueError if validation fails, listing ALL errors found.
    """
    errors = []

    # Check 1: All expected columns are present
    # checking if any expected columns from the golden schema are missing
    missing = set(schema["columns"]) - set(df.columns)
    if missing:
        errors.append(f"Missing columns: {missing}")

    # Check 2: Null rate per column is within tolerance
    # making sure null values are not higher than allowed tolerance
    for col in schema["columns"]:
        if col in df.columns:
            null_rate = df[col].isnull().mean()
            if null_rate > schema["null_tolerance"]:
                errors.append(f"{col}: null rate {null_rate:.1%} > tolerance")

    if errors:
        raise ValueError("Schema validation FAILED:\n" + "\n".join(f"   {e}" for e in errors))

    print(" Schema validation passed")
    return True


# ----- Test: v1 is the golden schema -----
golden = build_golden_schema(datasets["v1"])
print("Golden schema columns:", golden.get("columns", []))
print()

print("Validating v2 (has extra column)...")
try:
    validate_schema(datasets["v2"], golden)
except ValueError as e:
    print(f"Caught validation error:\n{e}")

print()
print("Validating v3 (has nulls + extra column)...")
try:
    validate_schema(datasets["v3"], golden)
except ValueError as e:
    print(f"  Caught validation error:\n{e}")

Golden schema columns: ['age', 'annual_income', 'loan_amount', 'credit_score', 'employment_years', 'debt_to_income', 'default']

Validating v2 (has extra column)...
 Schema validation passed

Validating v3 (has nulls + extra column)...
  Caught validation error:
Schema validation FAILED:
   credit_score: null rate 15.0% > tolerance


In [7]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    assert pointer_v1.get('md5') and len(pointer_v1['md5']) == 32, 'md5 must be a 32-char hex string'
    passed.append('DVC pointer created')
except Exception as e:
    failed.append(('DVC pointer created', str(e)))

try:
    assert all(k in pointer_v1 for k in ['md5','path','size','rows','cols']), 'Missing keys in pointer'
    passed.append('DVC pointer has required keys')
except Exception as e:
    failed.append(('DVC pointer has required keys', str(e)))

try:
    assert len(dvc_store) == 3, 'dvc_store should have v1, v2, v3'
    passed.append('All 3 versions stored')
except Exception as e:
    failed.append(('All 3 versions stored', str(e)))

try:
    assert r23.get('null_delta', 0) > 0, 'Should detect null increase in v3'
    passed.append('Drift detected v2→v3')
except Exception as e:
    failed.append(('Drift detected v2→v3', str(e)))

try:
    assert True, 'Manually verify the validator raised an error for v3'
    passed.append('Schema validator catches v3')
except Exception as e:
    failed.append(('Schema validator catches v3', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed: print(f'  {p}')
if failed:
    print('\nFailed:')
    for f,e in failed: print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')


 Passed: 5/5
  DVC pointer created
  DVC pointer has required keys
  All 3 versions stored
  Drift detected v2→v3
  Schema validator catches v3

All checks passed!


---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 1

Implement a complete DVC-style versioning system with the following requirements:

**Requirements:**
1. `create_dvc_pointer(path)` - MD5 checksum, full metadata, saved to `.dvc` file
2. `compare_dataset_versions(ptr_a, ptr_b)` - schema diff, row diff, null diff, per-column statistical drift (mean, std, min, max shifts)
3. `build_golden_schema(df)` + `validate_schema(df, schema)` - catches missing columns, null rate violations, and out-of-range values for numeric columns
4. `reproduce_dataset(version)` - given a version string, loads the correct CSV using the `.dvc` pointer file (simulating `dvc pull`)
5. Demonstrate: show that v3 fails validation, v1 passes, and `reproduce_dataset('v1')` returns the exact same checksum as the v1 pointer

Write all functions in the cell below with no scaffolding.

In [8]:
#  ADVANCED - Task 1 Full Implementation

import pandas as pd
import hashlib
import json
from pathlib import Path

DATA_DIR = LAB_DIR / "data"
DVC_DIR  = LAB_DIR / "dvc_store"
DATA_DIR.mkdir(exist_ok=True)
DVC_DIR.mkdir(exist_ok=True)
# ------------------------------------------------------
# 1. Create DVC pointer (checksum + metadata)
# ------------------------------------------------------
def create_dvc_pointer(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{path} does not exist")
    with open(path, "rb") as f:
        md5 = hashlib.md5(f.read()).hexdigest()

    df = pd.read_csv(path)

    metadata = {
        "file": path.name,
        "rows": len(df),
        "columns": list(df.columns),
        "checksum": md5
    }

    pointer = {
        "dataset": metadata,
        "path": str(path)
    }

    ptr_file = DVC_DIR / f"{path.stem}.dvc"
    with open(ptr_file, "w") as f:
        json.dump(pointer, f, indent=2)

    return pointer


# ------------------------------------------------------
# 2. Compare dataset versions
# ------------------------------------------------------
def compare_dataset_versions(ptr_a, ptr_b):

    df_a = pd.read_csv(ptr_a["path"])
    df_b = pd.read_csv(ptr_b["path"])

    print("Schema difference:")
    print(set(df_a.columns) ^ set(df_b.columns))

    print("\nRow count difference:")
    print(len(df_a) - len(df_b))

    print("\nNull differences:")
    print(df_a.isnull().sum() - df_b.isnull().sum())

    print("\nStatistical drift:")
    common_cols = set(df_a.columns).intersection(df_b.columns)

    for col in common_cols:
        if pd.api.types.is_numeric_dtype(df_a[col]):
            stats_a = df_a[col].describe()
            stats_b = df_b[col].describe()

            print(f"\nColumn: {col}")
            print("Mean shift:", stats_a["mean"] - stats_b["mean"])
            print("Std shift:", stats_a["std"] - stats_b["std"])
            print("Min shift:", stats_a["min"] - stats_b["min"])
            print("Max shift:", stats_a["max"] - stats_b["max"])


# ------------------------------------------------------
# 3. Build golden schema
# ------------------------------------------------------
def build_golden_schema(df):

    schema = {}

    for col in df.columns:

        col_data = df[col]

        schema[col] = {
            "dtype": str(col_data.dtype),
            "null_rate": col_data.isnull().mean()
        }

        if pd.api.types.is_numeric_dtype(col_data):
            schema[col]["min"] = float(col_data.min())
            schema[col]["max"] = float(col_data.max())

    return schema


# ------------------------------------------------------
# Validate schema
# ------------------------------------------------------
def validate_schema(df, schema):

    for col, rules in schema.items():

        if col not in df.columns:
            raise ValueError(f"Missing column: {col}")

        null_rate = df[col].isnull().mean()

        if null_rate > rules["null_rate"] + 0.05:
            raise ValueError(f"Null rate violation in {col}")

        if "min" in rules:
            if df[col].min() < rules["min"] or df[col].max() > rules["max"]:
                raise ValueError(f"Out of range values in {col}")

    print("Schema validation passed")


# ------------------------------------------------------
# 4. Reproduce dataset (simulate dvc pull)
# ------------------------------------------------------
def reproduce_dataset(version):

    ptr_file = DVC_DIR / f"{version}.dvc"

    with open(ptr_file) as f:
        pointer = json.load(f)

    df = pd.read_csv(pointer["path"])

    with open(pointer["path"], "rb") as f:
        md5 = hashlib.md5(f.read()).hexdigest()

    print("Checksum from pointer:", pointer["dataset"]["checksum"])
    print("Checksum from file:", md5)

    return df


# ------------------------------------------------------
# 5. Demonstration
# ------------------------------------------------------

# create pointers
ptr_v1 = create_dvc_pointer(DATA_DIR / "loans_v1.csv")
ptr_v3 = create_dvc_pointer(DATA_DIR / "loans_v3.csv")

# build schema from v1
df_v1 = pd.read_csv(DATA_DIR / "loans_v1.csv")
golden_schema = build_golden_schema(df_v1)

# validation
print("\nValidating v1")
validate_schema(df_v1, golden_schema)

print("\nValidating v3")
try:
    df_v3 = pd.read_csv(DATA_DIR / "loans_v3.csv")
    validate_schema(df_v3, golden_schema)
except Exception as e:
    print("Validation failed:", e)

# dataset comparison
print("\nDataset drift comparison:")
compare_dataset_versions(ptr_v1, ptr_v3)

# reproduce dataset
print("\nReproducing dataset v1")
df_reproduced = reproduce_dataset("loans_v1")





Validating v1
Schema validation passed

Validating v3
Validation failed: Out of range values in loan_amount

Dataset drift comparison:
Schema difference:
{'num_accounts'}

Row count difference:
0

Null differences:
age                   0.0
annual_income         0.0
credit_score       -300.0
debt_to_income        0.0
default               0.0
employment_years      0.0
loan_amount           0.0
num_accounts          NaN
dtype: float64

Statistical drift:

Column: loan_amount
Mean shift: -130.9724299999998
Std shift: -59.28757366845912
Min shift: 0.0
Max shift: -3498.510000000002

Column: employment_years
Mean shift: 0.2530000000000001
Std shift: -0.08765878534994975
Min shift: 0.0
Max shift: 0.0

Column: age
Mean shift: -0.26850000000000307
Std shift: -0.22937674624717275
Min shift: 0.0
Max shift: 0.0

Column: default
Mean shift: 0.034499999999999975
Std shift: -0.004403401361671722
Min shift: 0.0
Max shift: 0.0

Column: credit_score
Mean shift: -3.141644117647047
Std shift: 0.71876429

---
# TASK 2 - Experiment Tracking with MLflow
> Which model should go to production - and can you prove it?
---

##  Task 2 - Experiment Tracking with MLflow

### Scenario
> LoanGuard's data science lead trained three models last sprint - Logistic Regression, Random Forest, and Gradient Boosting. She can't remember which hyperparameters she used for the best one, and the comparison was done in a Slack message that got deleted. You need to run all three as tracked experiments so the decision is transparent and auditable.

### Objective
By the end of this task you will be able to:
- Create an MLflow experiment and log runs with parameters, metrics, and artifacts
- Compare multiple model runs programmatically
- Identify the best model using a business-relevant metric (F1 score for imbalanced default data)
- Log the training dataset version alongside model runs for full traceability

---

###  Background - MLflow Tracking

MLflow Tracking records every experiment run as a **first-class object** with:
- **Parameters** - hyperparameters, config values
- **Metrics** - accuracy, F1, AUC, etc.
- **Artifacts** - model files, plots, data samples
- **Tags** - free-form metadata (dataset version, author, etc.)

Each run gets a unique `run_id` that links it permanently to what produced it.

---

###  Choose Your Path

| Path | Description |
|------|-------------|
|  **Beginner** | Scaffolded code - fill in the blanks (`# YOUR CODE HERE`) |
|  **Advanced** | Open-ended - write from scratch using only the requirements |

**You are on the  Beginner path.** Skip to the  Advanced section if you want a challenge.

---

###  2A - Set up MLflow and prepare data

Configure MLflow tracking and split the versioned dataset into train/test sets.

In [9]:
#  BEGINNER 2A - Configure MLflow and prepare data
# ----------------------------------------------------

# Step 1: Set the MLflow tracking URI to your local SQLite database
# YOUR CODE HERE
# mlflow.set_tracking_uri(...)
mlflow.set_tracking_uri(MLFLOW_URI)
# Step 2: Create (or get) an experiment called "loandefault_prediction"
# YOUR CODE HERE
# experiment_id = mlflow.set_experiment(...)
experiment = mlflow.set_experiment("loandefault_prediction")
# Step 3: Load loans_v1.csv and split into train / test (80/20, random_state=42)
df = pd.read_csv(LAB_DIR / "data" / "loans_v1.csv").dropna()

FEATURES = ["age", "annual_income", "loan_amount",
            "credit_score", "employment_years", "debt_to_income"]
TARGET   = "default"

# YOUR CODE HERE
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Experiment ready")
print(f"Train size : {len(X_train) if X_train is not None else '?':,}")
print(f"Test size  : {len(X_test) if X_test is not None else '?':,}")
print(f"Default rate (train): {y_train.mean():.1%}" if y_train is not None else "")

2026/03/13 10:33:07 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/13 10:33:07 INFO mlflow.store.db.utils: Updating database tables
2026/03/13 10:33:11 INFO mlflow.tracking.fluent: Experiment with name 'loandefault_prediction' does not exist. Creating a new experiment.


Experiment ready
Train size : 1,600
Test size  : 400
Default rate (train): 57.6%


###  2B - Log a single MLflow run

Write a function that trains one model and logs everything to MLflow: params, metrics, the model artifact, and a tag for the dataset version.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `with mlflow.start_run(run_name=...) as run:` to start a run. Inside: `mlflow.log_param(key, val)` and `mlflow.log_metric(key, val)`.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Log the model with `mlflow.sklearn.log_model(model, artifact_path='model')`. Add tags with `mlflow.set_tag(key, val)`.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def run_experiment(model, model_name, params, X_tr, X_te, y_tr, y_te, dataset_version):
    with mlflow.start_run(run_name=model_name) as run:
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        proba = model.predict_proba(X_te)[:, 1]

        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy':  accuracy_score(y_te, preds),
            'precision': precision_score(y_te, preds),
            'recall':    recall_score(y_te, preds),
            'f1':        f1_score(y_te, preds),
            'auc':       roc_auc_score(y_te, proba)
        })
        mlflow.set_tag('dataset_version', dataset_version)
        mlflow.set_tag('model_type', model_name)
        mlflow.sklearn.log_model(model, 'model')
        return run.info.run_id, f1_score(y_te, preds)
```

</details>

In [10]:
#  BEGINNER 2B - Log a single MLflow run
# -----------------------------------------------------

def run_experiment(model, model_name, params, X_tr, X_te, y_tr, y_te, dataset_version):
    """
    Train a model and log a complete MLflow run.

    Must log:
      - All items in `params` dict as MLflow parameters
      - accuracy, precision, recall, f1, auc as MLflow metrics
      - The fitted model as an artifact
      - 'dataset_version' and 'model_type' as tags

    Returns
    -------
    (run_id : str, f1 : float)
    """
    with mlflow.start_run(run_name=model_name) as run:

        # Step 1: Train the model
        # YOUR CODE HERE
        model.fit(X_tr, y_tr)
        # Step 2: Generate predictions and probabilities
        # YOUR CODE HERE
        preds = model.predict(X_te)
        proba = model.predict_proba(X_te)[:, 1]

        # Step 3: Log parameters
        # YOUR CODE HERE

        # Step 4: Log metrics - accuracy, precision, recall, f1, auc
        # YOUR CODE HERE
        mlflow.log_params(params)
        # Step 5: Set tags for dataset_version and model_type
        # YOUR CODE HERE
        mlflow.log_metrics({
          "accuracy": accuracy_score(y_te, preds),
          "precision": precision_score(y_te, preds),
          "recall": recall_score(y_te, preds),
          "f1": f1_score(y_te, preds),
          "auc": roc_auc_score(y_te, proba)
        })
        mlflow.set_tag("dataset_version", dataset_version)
        mlflow.set_tag("model_type", model_name)
        # Step 6: Log the model artifact
        # YOUR CODE HERE
        mlflow.sklearn.log_model(model, "model")
        f1 = f1_score(y_te, preds) if preds is not None else 0.0
        return run.info.run_id, f1


# ----- Test with one model -----
test_model  = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
test_params = {"C": 1.0, "max_iter": 1000, "solver": "lbfgs"}
run_id, f1  = run_experiment(
    test_model, "LogisticRegression_test", test_params,
    X_train, X_test, y_train, y_test, dataset_version="v1"
)
print(f"Run ID : {run_id}")
print(f"F1     : {f1:.4f}")

2026/03/13 10:33:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 10:33:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run ID : aef8821b69fd483ab9c693c3321834b6
F1     : 0.8706


###  2C - Run all three candidate models

Run all three model types with the parameters below and collect their run IDs and F1 scores.

In [11]:
#  BEGINNER 2C - Run all three candidate models
# ------------------------------------------------------

# Three candidate models with their hyperparameter configs
candidates = [
    (
        LogisticRegression(C=0.5, max_iter=1000, random_state=42),
        "LogisticRegression",
        {"model": "LogisticRegression", "C": 0.5, "max_iter": 1000}
    ),
    (
        RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
        "RandomForest",
        {"model": "RandomForest", "n_estimators": 100, "max_depth": 6}
    ),
    (
        GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                   max_depth=4, random_state=42),
        "GradientBoosting",
        {"model": "GradientBoosting", "n_estimators": 100,
         "learning_rate": 0.1, "max_depth": 4}
    ),
]
# Store results as a list of dicts: [{"name": ..., "run_id": ..., "f1": ...}]
results = []

# YOUR CODE HERE - loop over candidates and call run_experiment for each
for model, name, params in candidates:
    run_id, f1 = run_experiment(
        model, name, params,
        X_train, X_test, y_train, y_test,
        dataset_version="v1"
    )

    results.append({
        "name": name,
        "run_id": run_id,
        "f1": f1
    })

# ----- Print comparison -----
print("\n" + "═"*55)
print(f"{'Model':<25} {'F1 Score':>10} {'Run ID':>15}")
print("═"*55)
for r in results:
    print(f"{r['name']:<25} {r['f1']:>10.4f}  {r['run_id'][:12]}...")
print("═"*55)

if results:
    best = max(results, key=lambda x: x["f1"])
    print(f"\nBest model: {best['name']}  (F1 = {best['f1']:.4f})")
    BEST_RUN_ID = best["run_id"]
    BEST_MODEL_NAME = best["name"]

2026/03/13 10:33:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 10:33:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 10:33:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 10:33:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p


═══════════════════════════════════════════════════════
Model                       F1 Score          Run ID
═══════════════════════════════════════════════════════
LogisticRegression            0.8706  0fb15cc544c4...
RandomForest                  0.8747  37035cf095ac...
GradientBoosting              0.8722  5ea198e55ae5...
═══════════════════════════════════════════════════════

Best model: RandomForest  (F1 = 0.8747)


###  2D - Retrieve the best model from MLflow

Use the MLflow client to load the best model back from the tracking server - as if you're another engineer picking up the work.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `MlflowClient()` to interact with the tracking server. `client.get_run(run_id)` retrieves run metadata.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

`mlflow.sklearn.load_model(f'runs:/{run_id}/model')` loads the model artifact from a run.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
client = MlflowClient()
run    = client.get_run(BEST_RUN_ID)
print(run.data.metrics)
model  = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")
preds  = model.predict(X_test)
```

</details>

In [12]:
#  BEGINNER 2D - Retrieve and verify the best model
# ------------------------------------------------------

client = MlflowClient()

# Step 1: Retrieve the run metadata for BEST_RUN_ID
# I fetch the run info using the run_id so we can inspect what was logged
best_run = client.get_run(BEST_RUN_ID)


# Step 2: Print what was logged
if best_run:
    print("Run metadata:")
    print(f"  Model type      : {best_run.data.tags.get('model_type')}")
    print(f"  Dataset version : {best_run.data.tags.get('dataset_version')}")
    print(f"  F1 score        : {best_run.data.metrics.get('f1', 0):.4f}")
    print(f"  Params          : {best_run.data.params}")

# Step 3: Load the model artifact
# Here I load the trained model saved in MLflow so it can be reused later
loaded_model = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")

# Step 4: Verify it produces the same predictions
if loaded_model is not None:
    reloaded_preds = loaded_model.predict(X_test)
    print(f"\nVerification F1: {f1_score(y_test, reloaded_preds):.4f}")
    print(" Model successfully retrieved from MLflow")

Run metadata:
  Model type      : RandomForest
  Dataset version : v1
  F1 score        : 0.8747
  Params          : {'model': 'RandomForest', 'n_estimators': '100', 'max_depth': '6'}



Verification F1: 0.8747
 Model successfully retrieved from MLflow


In [13]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    ## instead ' add
    assert mlflow.get_experiment_by_name('loandefault_prediction') is not None, "Create experiment 'loandefault_prediction'"
    passed.append('MLflow experiment exists')
except Exception as e:
    failed.append(('MLflow experiment exists', str(e)))

try:
    assert len(results) == 3, 'results list should have 3 entries'
    passed.append('3 candidates ran')
except Exception as e:
    failed.append(('3 candidates ran', str(e)))

try:
    assert 'BEST_RUN_ID' in dir() or 'BEST_RUN_ID' in globals(), 'BEST_RUN_ID must be defined'
    passed.append('Best run ID set')
except Exception as e:
    failed.append(('Best run ID set', str(e)))

try:
    assert loaded_model is not None, 'loaded_model should not be None'
    passed.append('Model reloads OK')
except Exception as e:
    failed.append(('Model reloads OK', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed: print(f'  {p}')
if failed:
    print('\nFailed:')
    for f,e in failed: print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')


 Passed: 4/4
  MLflow experiment exists
  3 candidates ran
  Best run ID set
  Model reloads OK

All checks passed!


---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 2

Implement a full experiment comparison pipeline:

**Requirements:**
1. Configure MLflow with the local SQLite URI
2. Write `run_experiment()` that logs: params, 5 metrics (accuracy/precision/recall/f1/auc), model artifact, dataset version tag, and training timestamp tag
3. Run all three candidate models (configurations given in Beginner 2C)
4. Write `find_best_run(experiment_name, metric='f1')` that queries MLflow programmatically (using `MlflowClient.search_runs()`) and returns the run with the highest value of `metric`
5. Load the best model, re-evaluate on test set, and confirm metrics match what was logged
6. **Bonus:** Add a second experiment round where you retrain the best model type on `loans_v2.csv` and log the dataset version change

In [14]:
"""
I got  this so i did

 2026/03/11 16:42:46 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/11 16:42:46 INFO mlflow.store.db.utils: Updating database tables
2026/03/11 16:42:50 INFO mlflow.tracking.fluent: Experiment with name 'loandefault_prediction' does not exist. Creating a new experiment.
MLflow experiment ready
Dataset ready
Train size: 1600
Test size : 400
2026/03/11 16:42:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/11 16:42:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/11 16:42:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/11 16:42:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/11 16:43:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/11 16:43:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html

Model Comparison
{'name': 'LogisticRegression', 'run_id': 'fe4399a11b9a4a37a4d8337f4a62de22', 'f1': 0.8706365503080082}
{'name': 'RandomForest', 'run_id': '310d3d3cb73c4c67854a6e46fee65bff', 'f1': 0.8747433264887063}
{'name': 'GradientBoosting', 'run_id': '27acc034c47a44d886f00f950789546f', 'f1': 0.8722109533468559}

Best run found:
Run ID: 310d3d3cb73c4c67854a6e46fee65bff
F1: 0.8747433264887063
Downloading artifacts:   0% 0/1 [00:00<?, ?it/s]Downloading artifacts: 100% 5/5 [00:00<00:00, 243.22it/s]
Reloaded model F1: 0.8747433264887063

Running second experiment with dataset v2
2026/03/11 16:43:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/11 16:43:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
('c22dcbf33b4842d8aedb1395642f76e6', 0.8747433264887063)
"""

import warnings
import logging

# Hide Python warnings
warnings.filterwarnings("ignore")

# Suppress MLflow logs
logging.getLogger("mlflow").setLevel(logging.ERROR)

# Optional: suppress sklearn warnings too
logging.getLogger("sklearn").setLevel(logging.ERROR)


In [15]:
#  ADVANCED - Task 2 Full Implementation
# ----------------------------------------------------------

# YOUR CODE HERE



# ---------------------------------------
# 1. Configure MLflow tracking
# ---------------------------------------

# Using a local SQLite database so all runs are saved persistently
mlflow.set_tracking_uri("sqlite:///mlruns.db")

# Create or access experiment
experiment = mlflow.set_experiment("loandefault_prediction")

client = MlflowClient()

print("MLflow experiment ready")


# ---------------------------------------
# 2. Prepare dataset
# ---------------------------------------

# Loading dataset and removing rows with nulls
df = pd.read_csv(LAB_DIR / "data" / "loans_v1.csv").dropna()

FEATURES = [
    "age","annual_income","loan_amount",
    "credit_score","employment_years","debt_to_income"
]

TARGET = "default"

X = df[FEATURES]
y = df[TARGET]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Dataset ready")
print("Train size:", len(X_train))
print("Test size :", len(X_test))


# ---------------------------------------
# 3. Experiment run function
# ---------------------------------------

def run_experiment(model, model_name, params,
                   X_tr, X_te, y_tr, y_te,
                   dataset_version):

    # Starting an MLflow run so everything gets tracked
    with mlflow.start_run(run_name=model_name) as run:

        # Train model
        model.fit(X_tr, y_tr)

        # Predictions
        preds = model.predict(X_te)
        proba = model.predict_proba(X_te)[:,1]

        # Calculate metrics
        acc  = accuracy_score(y_te, preds)
        prec = precision_score(y_te, preds)
        rec  = recall_score(y_te, preds)
        f1   = f1_score(y_te, preds)
        auc  = roc_auc_score(y_te, proba)

        # Log parameters
        mlflow.log_params(params)

        # Log metrics
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("auc", auc)

        # Tags help with traceability
        mlflow.set_tag("dataset_version", dataset_version)
        mlflow.set_tag("model_type", model_name)
        mlflow.set_tag("training_time", datetime.now().isoformat())

        # Save model artifact
        mlflow.sklearn.log_model(model, "model")

        return run.info.run_id, f1


# ---------------------------------------
# 4. Candidate models
# ---------------------------------------

candidates = [

    (
        LogisticRegression(C=0.5, max_iter=1000, random_state=42),
        "LogisticRegression",
        {"C":0.5,"max_iter":1000}
    ),

    (
        RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
        "RandomForest",
        {"n_estimators":100,"max_depth":6}
    ),

    (
        GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=4,
            random_state=42
        ),
        "GradientBoosting",
        {"n_estimators":100,"learning_rate":0.1,"max_depth":4}
    )
]


# ---------------------------------------
# 5. Run experiments
# ---------------------------------------

results = []

for model, name, params in candidates:

    run_id, f1 = run_experiment(
        model,
        name,
        params,
        X_train,
        X_test,
        y_train,
        y_test,
        dataset_version="v1"
    )

    results.append({
        "name":name,
        "run_id":run_id,
        "f1":f1
    })


# Print comparison
print("\nModel Comparison")
for r in results:
    print(r)


# ---------------------------------------
# 6. Find best run programmatically
# ---------------------------------------

def find_best_run(experiment_name, metric="f1"):

    exp = client.get_experiment_by_name(experiment_name)

    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=[f"metrics.{metric} DESC"]
    )

    best_run = runs[0]

    print("\nBest run found:")
    print("Run ID:", best_run.info.run_id)
    print("F1:", best_run.data.metrics.get(metric))

    return best_run


best_run = find_best_run("loandefault_prediction")

BEST_RUN_ID = best_run.info.run_id


# ---------------------------------------
# 7. Load best model
# ---------------------------------------

# Loading saved model artifact from MLflow
best_model = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")

preds = best_model.predict(X_test)

print("\nReloaded model F1:",
      f1_score(y_test, preds))


# ---------------------------------------
# 8. BONUS: retrain best model on dataset v2
# ---------------------------------------

df_v2 = pd.read_csv(LAB_DIR / "data" / "loans_v2.csv").dropna()

X2 = df_v2[FEATURES]
y2 = df_v2[TARGET]

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

# Retrain using the best model type
best_type = best_run.data.tags["model_type"]

if best_type == "RandomForest":
    model = RandomForestClassifier(n_estimators=100,max_depth=6)

elif best_type == "GradientBoosting":
    model = GradientBoostingClassifier(n_estimators=100)

else:
    model = LogisticRegression(max_iter=1000)


print("\nRunning second experiment with dataset v2")

run_experiment(
    model,
    best_type+"_v2",
    {},
    X_train2,
    X_test2,
    y_train2,
    y_test2,
    dataset_version="v2"
)












MLflow experiment ready
Dataset ready
Train size: 1600
Test size : 400

Model Comparison
{'name': 'LogisticRegression', 'run_id': 'd5da9b06ebf24ffcba409396d8f7966e', 'f1': 0.8706365503080082}
{'name': 'RandomForest', 'run_id': '06847b87d51243f2b84dea1ed2ff6945', 'f1': 0.8747433264887063}
{'name': 'GradientBoosting', 'run_id': '235ff936487146bf9210a79f3152504d', 'f1': 0.8722109533468559}

Best run found:
Run ID: 06847b87d51243f2b84dea1ed2ff6945
F1: 0.8747433264887063



Reloaded model F1: 0.8747433264887063

Running second experiment with dataset v2


('6e54c88d18804f7db00300643fbee59f', 0.8747433264887063)

---
# TASK 3 - Model Versioning
> Give your model a name that tells the whole story
---

##  Task 3 - Model Versioning - Naming, Metadata & Semantic Versions

### Scenario
> LoanGuard's best model from Task 2 needs to be packaged for the team. The previous engineer just saved files as `model.pkl` with no metadata. When it broke, nobody knew what version was running or how to go back. You need to create a versioned model artifact with full metadata and implement semantic versioning logic.

### Objective
By the end of this task you will be able to:
- Save a model artifact with complete metadata (dataset version, metrics, git hash, author)
- Implement semantic versioning logic (MAJOR.MINOR.PATCH) for model releases
- Register a named, versioned model in MLflow Model Registry
- Retrieve a specific model version by name

---

###  Choose Your Path

| Path | Description |
|------|-------------|
|  **Beginner** | Scaffolded code - fill in the blanks (`# YOUR CODE HERE`) |
|  **Advanced** | Open-ended - write from scratch using only the requirements |

**You are on the  Beginner path.** Skip to the  Advanced section if you want a challenge.

---

###  3A - Save a model with complete metadata

Save the best model from Task 2 alongside a metadata sidecar file that fully describes it.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Save the model with `pickle.dump()`. Create a separate `_metadata.json` file in the same folder with all relevant context.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Metadata should include: model_name, version, dataset_version, metrics, params, trained_at, and a simulated git_hash (`hashlib.md5(str(time.time()).encode()).hexdigest()[:8]`).

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def save_versioned_model(model, name, version, metrics, params, dataset_version):
    path = LAB_DIR / 'models' / f'{name}-{version}'
    path.mkdir(exist_ok=True)
    with open(path / 'model.pkl', 'wb') as f:
        pickle.dump(model, f)
    meta = {
        'name': name, 'version': version,
        'dataset_version': dataset_version,
        'metrics': metrics, 'params': params,
        'git_hash': hashlib.md5(str(time.time()).encode()).hexdigest()[:8],
        'trained_at': datetime.now().isoformat(),
        'mlflow_run_id': BEST_RUN_ID
    }
    with open(path / 'metadata.json', 'w') as f:
        json.dump(meta, f, indent=2)
    return path, meta
```

</details>

In [16]:
#  BEGINNER 3A - Save versioned model with metadata
# ------------------------------------------------------------

def save_versioned_model(model, name, version, metrics, params,
                          dataset_version, run_id):
    """
    Save a model and its complete metadata sidecar.

    Creates:
      models/{name}-{version}/
        model.pkl        ← serialised model
        metadata.json    ← full provenance record

    Returns
    -------
    (path : Path, metadata : dict)
    """
    # Step 1: Create output directory
    path = LAB_DIR / "models" / f"{name}-{version}"
    # YOUR CODE HERE
    path.mkdir(parents=True, exist_ok=True)   # create folder if it doesn't exist

    # Step 2: Save model with pickle
    with open(path / "model.pkl", "wb") as f:
        pickle.dump(model, f)


    # Step 3: Build metadata dict
    metadata = {
        "name":             name,
        "version":          version,
        "dataset_version":  dataset_version,
        "metrics":          metrics,
        "params":           params,
        "git_hash":         hashlib.md5(str(time.time()).encode()).hexdigest()[:8],
        "trained_at":       datetime.now().isoformat(),
        "mlflow_run_id":    run_id,
        "author":           "loanGuard-ml-team"
    }

    # Step 4: Save metadata as JSON
    # This file stores all info about how the model was created
    with open(path / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    return path, metadata

# ----- Save the best model from Task 2 -----
best_run_data   = MlflowClient().get_run(BEST_RUN_ID)
best_metrics    = best_run_data.data.metrics
best_params     = best_run_data.data.params
best_fitted     = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")

model_path, model_meta = save_versioned_model(
    model           = best_fitted,
    name            = "loan-default-detector",
    version         = "v1.0.0",
    metrics         = best_metrics,
    params          = best_params,
    dataset_version = "v1",
    run_id          = BEST_RUN_ID
)

print(f"Saved to: {model_path}")
print("\nMetadata:")
print(json.dumps(model_meta, indent=2))

Saved to: loanGuard_lab/models/loan-default-detector-v1.0.0

Metadata:
{
  "name": "loan-default-detector",
  "version": "v1.0.0",
  "dataset_version": "v1",
  "metrics": {
    "accuracy": 0.8475,
    "precision": 0.8623481781376519,
    "recall": 0.8875,
    "f1": 0.8747433264887063,
    "auc": 0.9296354166666667
  },
  "params": {
    "n_estimators": "100",
    "max_depth": "6"
  },
  "git_hash": "bcb3d2ea",
  "trained_at": "2026-03-13T10:33:55.072446",
  "mlflow_run_id": "06847b87d51243f2b84dea1ed2ff6945",
  "author": "loanGuard-ml-team"
}


###  3B - Semantic versioning logic

Implement a function that determines the correct next version number given the type of change.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Parse the version string by splitting on '.': `major, minor, patch = version.lstrip('v').split('.')`.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

For 'major' bump: increment major, reset minor and patch to 0. For 'minor': increment minor, reset patch. For 'patch': just increment patch.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def bump_version(current, bump_type):
    major, minor, patch = [int(x) for x in current.lstrip('v').split('.')]
    if bump_type == 'major':   return f'v{major+1}.0.0'
    elif bump_type == 'minor': return f'v{major}.{minor+1}.0'
    elif bump_type == 'patch': return f'v{major}.{minor}.{patch+1}'
```

</details>

In [17]:
#  BEGINNER 3B - Semantic versioning
# ------------------------------------------------------------

def bump_version(current_version, bump_type):
    """
    Increment a semantic version string.

    Parameters
    ----------
    current_version : str  e.g. 'v1.2.3'
    bump_type       : str  one of 'major', 'minor', 'patch'

    Returns
    -------
    str  e.g. 'v2.0.0' for a major bump on 'v1.2.3'

    Rules
    -----
    major → increment MAJOR, reset MINOR and PATCH to 0
    minor → increment MINOR, reset PATCH to 0
    patch → increment PATCH only
    """
    # Step 1: Parse current_version into major, minor, patch integers
    # YOUR CODE HERE
    major, minor, patch = 0, 0, 0
    # changed to
    # remove 'v' then split by '.'
    major, minor, patch = [int(x) for x in current_version.lstrip('v').split('.')]

    # Step 2: Apply bump logic
    # YOUR CODE HERE
        # simple rules based on semantic versioning
    if bump_type == "major":
        major += 1
        minor = 0
        patch = 0

    elif bump_type == "minor":
        minor += 1
        patch = 0

    elif bump_type == "patch":
        patch += 1

    return f"v{major}.{minor}.{patch}"


# ----- Test all bump types -----
start = "v1.0.0"
print(f"Start version      : {start}")
print(f"After patch bump   : {bump_version(start, 'patch')}   ← threshold tweak")
print(f"After minor bump   : {bump_version(start, 'minor')}   ← new training data")
print(f"After major bump   : {bump_version(start, 'major')}   ← new architecture")
print()

# Simulate LoanGuard's version history
history = [
    ("v1.0.0", "Initial release - Random Forest on loans_v1"),
    (bump_version("v1.0.0", "patch"), "Fixed decision threshold from 0.5 → 0.46"),
    (bump_version("v1.0.1", "minor"), "Retrained on loans_v2 with new num_accounts feature"),
    (bump_version("v1.1.0", "major"), "Switched to GradientBoosting - breaking output format change"),
]
print("LoanGuard model version history:")
for ver, desc in history:
    print(f"  {ver:<12} {desc}")

Start version      : v1.0.0
After patch bump   : v1.0.1   ← threshold tweak
After minor bump   : v1.1.0   ← new training data
After major bump   : v2.0.0   ← new architecture

LoanGuard model version history:
  v1.0.0       Initial release - Random Forest on loans_v1
  v1.0.1       Fixed decision threshold from 0.5 → 0.46
  v1.1.0       Retrained on loans_v2 with new num_accounts feature
  v2.0.0       Switched to GradientBoosting - breaking output format change


###  3C - Register in MLflow Model Registry

Register your best model in the MLflow Model Registry under a named model, then retrieve it by name.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `mlflow.register_model(model_uri, name)` where `model_uri = f'runs:/{run_id}/model'`.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

After registration, use `MlflowClient().get_registered_model(name)` to inspect it. `client.get_latest_versions(name)` lists all versions.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
model_uri = f"runs:/{BEST_RUN_ID}/model"
reg = mlflow.register_model(model_uri, "LoanDefaultDetector")
print(reg.version, reg.name)

client  = MlflowClient()
version = client.get_latest_versions("LoanDefaultDetector")[0]
loaded  = mlflow.sklearn.load_model(f"models:/LoanDefaultDetector/{version.version}")
```

</details>

In [18]:
#  BEGINNER 3C - Register model in MLflow Registry
# ----------------------------------------------------------

REGISTRY_NAME = "LoanDefaultDetector"

# Step 1: Register the best model run in the MLflow registry
# I register the model using the run artifact path
model_uri = f"runs:/{BEST_RUN_ID}/model"
registered = mlflow.register_model(model_uri, REGISTRY_NAME)

if registered:
    print(f"Registered model : {registered.name}")
    print(f"Version          : {registered.version}")
    print(f"Source run       : {registered.source}")

# Step 2: Retrieve and inspect all registered versions
# Using MLflow client to see all versions stored in registry
client = MlflowClient()
versions = client.get_latest_versions(REGISTRY_NAME)

for v in versions:
    print(f"Model: {v.name}  |  Version: {v.version}  |  Stage: {v.current_stage}")


# Step 3: Load the registered model by name and verify
# Loading the model directly from the registry instead of run_id
latest_version = versions[0]

registry_model = mlflow.sklearn.load_model(
    f"models:/{REGISTRY_NAME}/{latest_version.version}"
)



if registry_model is not None:
    verify_preds = registry_model.predict(X_test)
    print(f"\nVerification F1 from registry: {f1_score(y_test, verify_preds):.4f}")
    print(" Model successfully retrieved from registry by name")

Registered model : LoanDefaultDetector
Version          : 1
Source run       : models:/m-ab18bb3558db497cbfdcd1d42efcbc91
Model: LoanDefaultDetector  |  Version: 1  |  Stage: None

Verification F1 from registry: 0.8747
 Model successfully retrieved from registry by name


Successfully registered model 'LoanDefaultDetector'.
Created version '1' of model 'LoanDefaultDetector'.


In [19]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    assert (LAB_DIR / 'models' / 'loan-default-detector-v1.0.0' / 'model.pkl').exists(), 'model.pkl not found'
    passed.append('Versioned model saved')
except Exception as e:
    failed.append(('Versioned model saved', str(e)))

try:
    assert (LAB_DIR / 'models' / 'loan-default-detector-v1.0.0' / 'metadata.json').exists(), 'metadata.json not found'
    passed.append('Metadata saved')
except Exception as e:
    failed.append(('Metadata saved', str(e)))

try:
    assert bump_version('v1.0.0', 'patch') == 'v1.0.1', 'patch bump failed'
    passed.append('Patch bump correct')
except Exception as e:
    failed.append(('Patch bump correct', str(e)))

try:
    assert bump_version('v1.0.0', 'minor') == 'v1.1.0', 'minor bump failed'
    passed.append('Minor bump correct')
except Exception as e:
    failed.append(('Minor bump correct', str(e)))

try:
    assert bump_version('v1.0.0', 'major') == 'v2.0.0', 'major bump failed'
    passed.append('Major bump correct')
except Exception as e:
    failed.append(('Major bump correct', str(e)))

try:
    assert registered is not None, 'registered model should not be None'
    passed.append('Model registered')
except Exception as e:
    failed.append(('Model registered', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed: print(f'  {p}')
if failed:
    print('\nFailed:')
    for f,e in failed: print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')


 Passed: 6/6
  Versioned model saved
  Metadata saved
  Patch bump correct
  Minor bump correct
  Major bump correct
  Model registered

All checks passed!


---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 3

Build a complete model versioning and registry system:

**Requirements:**
1. `save_versioned_model()` - saves model + full metadata sidecar (all fields from Beginner 3A)
2. `bump_version()` - semantic version logic with validation (reject unknown bump types)
3. `ModelVersionHistory` class with methods:
   - `record(version, change_type, description, metrics)` - adds an entry
   - `get_changelog()` - returns formatted changelog string
   - `get_version_for_rollback(n_versions_back)` - returns the version string from n releases ago
4. Register model in MLflow Registry, add a description, and add the semantic version as a tag
5. **Demonstrate rollback:** Simulate a production failure, use `get_version_for_rollback(1)` to identify the previous version, load it from the local store, and confirm it produces different (better) metrics than the failed version

In [20]:
#  ADVANCED - Task 3 Full Implementation
# ----------------------------------------------------------

# YOUR CODE HERE



#Model Explanation
# 1. Model Saving with Metadata
# Saves trained models locally as model.pkl.
# Creates metadata.json with name, version, metrics, params, timestamp, and author.
# 2. Semantic Versioning
# Uses v<major>.<minor>.<patch> format.
# bump_version automatically increments major, minor, or patch versions.
# 3. Version History Tracking
# ModelVersionHistory logs every version and change.
# Records version, change type, description, and performance metrics.
# Changelog can be printed for a quick overview.
# 4. MLflow Model Registry
# Registers models in MLflow for centralized storage.
# Adds description and semantic version tags for clarity.
# 5. Rollback and Backup
# Can rollback to previous stable versions if performance drops.
# Local copy ensures offline access and traceability.
# 6. Workflow Summary
# Tracks model evolution like software version control.
# Helps maintain stable production models with proper documentation.

#  ADVANCED - Task 3 Full Implementation
# ----------------------------------------------------------

import pickle
import json
import hashlib
import time
from datetime import datetime
from pathlib import Path

import mlflow
from mlflow.tracking import MlflowClient


# ----------------------------------------------------------
# 1. Save model with full metadata
# ----------------------------------------------------------

def save_versioned_model(model, name, version, metrics, params,
                         dataset_version, run_id):

    # create folder for this model version
    path = LAB_DIR / "models" / f"{name}-{version}"
    path.mkdir(parents=True, exist_ok=True)

    # save the model
    with open(path / "model.pkl", "wb") as f:
        pickle.dump(model, f)

    # create metadata sidecar file
    metadata = {
        "name": name,
        "version": version,
        "dataset_version": dataset_version,
        "metrics": metrics,
        "params": params,
        "git_hash": hashlib.md5(str(time.time()).encode()).hexdigest()[:8],
        "trained_at": datetime.now().isoformat(),
        "mlflow_run_id": run_id,
        "author": "loanGuard-ml-team"
    }

    # save metadata
    with open(path / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    return path, metadata


# ----------------------------------------------------------
# 2. Semantic versioning with validation
# ----------------------------------------------------------

def bump_version(current_version, bump_type):

    if bump_type not in ["major", "minor", "patch"]:
        raise ValueError("Invalid bump type")

    major, minor, patch = [int(x) for x in current_version.lstrip("v").split(".")]

    if bump_type == "major":
        major += 1
        minor = 0
        patch = 0

    elif bump_type == "minor":
        minor += 1
        patch = 0

    elif bump_type == "patch":
        patch += 1

    return f"v{major}.{minor}.{patch}"


# ----------------------------------------------------------
# 3. Model Version History class
# ----------------------------------------------------------

class ModelVersionHistory:

    def __init__(self):
        self.history = []

    def record(self, version, change_type, description, metrics):

        entry = {
            "version": version,
            "change_type": change_type,
            "description": description,
            "metrics": metrics,
            "time": datetime.now().isoformat()
        }

        self.history.append(entry)

    def get_changelog(self):

        lines = ["Model Changelog\n"]

        for h in self.history:
            lines.append(
                f"{h['version']} | {h['change_type']} | "
                f"{h['description']} | F1={h['metrics'].get('f1',0):.4f}"
            )

        return "\n".join(lines)

    def get_version_for_rollback(self, n_versions_back):

        if n_versions_back >= len(self.history):
            raise ValueError("Not enough versions for rollback")

        return self.history[-(n_versions_back+1)]["version"]


# ----------------------------------------------------------
# 4. Register model in MLflow Registry
# ----------------------------------------------------------

REGISTRY_NAME = "LoanDefaultDetector"

model_uri = f"runs:/{BEST_RUN_ID}/model"

registered = mlflow.register_model(model_uri, REGISTRY_NAME)

client = MlflowClient()

# add description
client.update_registered_model(
    name=REGISTRY_NAME,
    description="Loan default prediction model for LoanGuard"
)

# add semantic version tag
client.set_model_version_tag(
    name=REGISTRY_NAME,
    version=registered.version,
    key="semantic_version",
    value="v1.0.0"
)


# ----------------------------------------------------------
# 5. Create version history
# ----------------------------------------------------------

history = ModelVersionHistory()

best_run_data = client.get_run(BEST_RUN_ID)

history.record(
    "v1.0.0",
    "major",
    "Initial production model",
    best_run_data.data.metrics
)


# ----------------------------------------------------------
# 6. Save the model locally
# ----------------------------------------------------------

best_model = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")

model_path, meta = save_versioned_model(
    model=best_model,
    name="loan-default-detector",
    version="v1.0.0",
    metrics=best_run_data.data.metrics,
    params=best_run_data.data.params,
    dataset_version="v1",
    run_id=BEST_RUN_ID
)


# ----------------------------------------------------------
# 7. Simulate a new version (bad model)
# ----------------------------------------------------------

bad_version = bump_version("v1.0.0", "patch")

history.record(
    bad_version,
    "patch",
    "Changed decision threshold (caused performance drop)",
    {"f1": 0.70}
)


# ----------------------------------------------------------
# 8. Demonstrate rollback
# ----------------------------------------------------------

print("\nSimulating production failure...")

rollback_version = history.get_version_for_rollback(1)

print("Rolling back to:", rollback_version)

rollback_path = LAB_DIR / "models" / f"loan-default-detector-{rollback_version}"

with open(rollback_path / "model.pkl", "rb") as f:
    rollback_model = pickle.load(f)


# evaluate rollback model
rollback_preds = rollback_model.predict(X_test)

rollback_f1 = f1_score(y_test, rollback_preds)

print("Rollback model F1:", rollback_f1)


# ----------------------------------------------------------
# 9. Print changelog
# ----------------------------------------------------------

print("\n", history.get_changelog())

Registered model 'LoanDefaultDetector' already exists. Creating a new version of this model...
Created version '2' of model 'LoanDefaultDetector'.



Simulating production failure...
Rolling back to: v1.0.0
Rollback model F1: 0.8747433264887063

 Model Changelog

v1.0.0 | major | Initial production model | F1=0.8747
v1.0.1 | patch | Changed decision threshold (caused performance drop) | F1=0.7000


---
# TASK 4 - Registry Lifecycle Management
> Staging → Production → Archived: govern your models
---

##  Task 4 - Registry Lifecycle - Staging, Production, Archived

### Scenario
> LoanGuard just hired a new senior ML engineer. She asks: 'What model is in production right now? When was it approved? Who approved it? What does rollback look like?' Nobody can answer. In this task you'll implement the full governance lifecycle - every model moves through defined stages with approvals and audit logs.

### Objective
By the end of this task you will be able to:
- Transition a model through Staging → Production → Archived stages in MLflow
- Write an audit log that records every lifecycle event
- Implement a rollback function that demotes the current production model
- Query the registry to answer governance questions programmatically

---

###  Choose Your Path

| Path | Description |
|------|-------------|
|  **Beginner** | Scaffolded code - fill in the blanks (`# YOUR CODE HERE`) |
|  **Advanced** | Open-ended - write from scratch using only the requirements |

**You are on the  Beginner path.** Skip to the  Advanced section if you want a challenge.

---

###  4A - Transition model through lifecycle stages

Move the registered model through the stages: None → Staging → Production

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `MlflowClient().transition_model_version_stage(name, version, stage)` where stage is 'Staging', 'Production', or 'Archived'.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

After each transition, add a comment: `client.update_model_version(name, version, description=...)` to record the reason.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
client = MlflowClient()
client.transition_model_version_stage(
    name    = REGISTRY_NAME,
    version = "1",
    stage   = "Staging",
    archive_existing_versions=False
)
```

</details>

In [21]:
#  BEGINNER 4A - Lifecycle stage transitions
# ---------------------------------------------------------

client = MlflowClient()

def transition_stage(name, version, stage, reason=""):
    """
    Transition a registered model version to a new stage and record the reason.

    Parameters
    ----------
    name    : str  registered model name
    version : str  model version number as string e.g. '1'
    stage   : str  'Staging', 'Production', or 'Archived'
    reason  : str  human-readable reason for the transition
    """
    # Step 1: Transition the model stage
    client.transition_model_version_stage(
        name=name,
        version=version,
        stage=stage,
        archive_existing_versions=(stage == "Production")  # optional: auto-archive old production
    )

    # Step 2: Update the model version description with the reason
    client.update_model_version(
        name=name,
        version=version,
        description=reason
    )

    print(f"  {name} v{version}  →  {stage}")
    if reason:
        print(f"Reason: {reason}")


# ----- Walk the model through its lifecycle -----
latest = client.get_latest_versions(REGISTRY_NAME)
if latest:
    ver = latest[0].version

    print("Stage transitions for LoanDefaultDetector:")
    print("─" * 50)

    transition_stage(
        REGISTRY_NAME, ver, "Staging",
        reason="Passed unit tests and offline evaluation - F1 > 0.72 threshold met"
    )

    # YOUR CODE HERE - transition to Production with a reason
    # Move to Production next
    transition_stage(
        REGISTRY_NAME, ver, "Production",
        reason="Approved by ML lead after review of evaluation metrics and QA tests"
    )

    # Verify current stage
    current = client.get_model_version(REGISTRY_NAME, ver)
    print(f"\nCurrent stage: {current.current_stage}")
    print(f"Description  : {current.description}")

Stage transitions for LoanDefaultDetector:
──────────────────────────────────────────────────
  LoanDefaultDetector v2  →  Staging
Reason: Passed unit tests and offline evaluation - F1 > 0.72 threshold met
  LoanDefaultDetector v2  →  Production
Reason: Approved by ML lead after review of evaluation metrics and QA tests

Current stage: Production
Description  : Approved by ML lead after review of evaluation metrics and QA tests


###  4B - Build an audit log

Every lifecycle event should be recorded. Implement an audit log that tracks: timestamp, model name, version, old stage, new stage, and the actor/reason.

In [22]:
#  BEGINNER 4B - Audit log
# ---------------------------------------------------------

AUDIT_LOG = []   # list of event dicts

def log_audit_event(model_name, version, from_stage, to_stage,
                     actor, reason):
    """
    Record a lifecycle event in the audit log and persist to disk.
    """
    event = {
        "timestamp":    datetime.now().isoformat(),
        "model_name":   model_name,
        "version":      version,
        "from_stage":   from_stage,
        "to_stage":     to_stage,
        "actor":        actor,
        "reason":       reason
    }

    # Step 1: Append to in-memory log
    AUDIT_LOG.append(event)

    # Step 2: Save to JSON file (append-style)
    log_path = LAB_DIR / "audit_log.json"
    if log_path.exists():
        with open(log_path, "r") as f:
            existing = json.load(f)
    else:
        existing = []

    existing.append(event)

    with open(log_path, "w") as f:
        json.dump(existing, f, indent=2)

    return event


def print_audit_log():
    """Print a formatted audit log table."""
    if not AUDIT_LOG:
        print("Audit log is empty")
        return
    print(f"{'Timestamp':<26} {'Model':<22} {'v':<4} {'From':<12} {'To':<12} {'Actor':<18} Reason")
    print("─" * 110)
    for e in AUDIT_LOG:
        ts = e['timestamp'][:19]
        print(f"{ts:<26} {e['model_name']:<22} {e['version']:<4} "
              f"{e['from_stage']:<12} {e['to_stage']:<12} {e['actor']:<18} {e['reason']}")


# ----- Replay lifecycle events through audit log -----
ver = client.get_latest_versions(REGISTRY_NAME)[0].version

log_audit_event(REGISTRY_NAME, ver, "None", "Staging",
                "ci-system", "Automated: offline eval passed F1 threshold")
log_audit_event(REGISTRY_NAME, ver, "Staging", "Production",
                "ml-lead@loanGuard.ai", "Manual approval: A/B test shows +3% approval lift")

print("LoanGuard Model Audit Log:")
print("=" * 110)
print_audit_log()

LoanGuard Model Audit Log:
Timestamp                  Model                  v    From         To           Actor              Reason
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
2026-03-13T10:33:55        LoanDefaultDetector    1    None         Staging      ci-system          Automated: offline eval passed F1 threshold
2026-03-13T10:33:55        LoanDefaultDetector    1    Staging      Production   ml-lead@loanGuard.ai Manual approval: A/B test shows +3% approval lift


###  4C - Rollback

Simulate a production failure: a new model version is promoted, starts failing, and you need to roll back to the previous production model.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

To rollback: archive the current production version, then transition the previous version back to Production.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Use `client.search_model_versions(f"name='{name}'")` to find all versions. Filter by `current_stage == 'Archived'` to find previous production candidates.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def rollback_production(name, rollback_to_version, reason):
    # 1. Find and archive current production version
    current_prod = [v for v in client.search_model_versions(f"name='{name}'")
                    if v.current_stage == 'Production']
    if current_prod:
        client.transition_model_version_stage(name, current_prod[0].version, 'Archived')
    # 2. Promote rollback target to Production
    client.transition_model_version_stage(name, rollback_to_version, 'Production')
```

</details>

In [23]:
#  BEGINNER 4C - Simulate promotion of v2, then rollback
# ---------------------------------------------------------

# First: Register a second (worse) model version to simulate a bad deployment
print("Simulating a bad deployment...")

# Train a deliberately weak model
weak_model = LogisticRegression(C=0.001, max_iter=100, random_state=42)
with mlflow.start_run(run_name="WeakModel_bad_deploy") as bad_run:
    weak_model.fit(X_train, y_train)
    weak_preds = weak_model.predict(X_test)
    mlflow.log_metric("f1", f1_score(y_test, weak_preds))
    mlflow.set_tag("dataset_version", "v1")
    mlflow.sklearn.log_model(weak_model, "model")
    bad_run_id = bad_run.info.run_id

# Register the weak model as version 2
bad_registered = mlflow.register_model(f"runs:/{bad_run_id}/model", REGISTRY_NAME)
bad_version     = bad_registered.version

# Promote bad model to Production (simulating a mistaken deployment)
client.transition_model_version_stage(REGISTRY_NAME, bad_version, "Production",
                                       archive_existing_versions=True)
log_audit_event(REGISTRY_NAME, bad_version, "Staging", "Production",
                "auto-deploy-pipeline", "Automated promotion - metric check bypassed (BUG)")

print(f"Bad model (version {bad_version}) promoted to Production - F1: {f1_score(y_test, weak_preds):.4f}")
print("  Production is now degraded!")

print()
print("─" * 50)
print("Initiating rollback...")
print("─" * 50)


def rollback_production(registry_name, rollback_to_version, reason, actor):
    """
    Roll back production to a previously archived version.

    Steps:
      1. Find the current Production version and move it to Archived
      2. Move rollback_to_version back to Production
      3. Log both events in the audit log
    """
    # Step 1: Find current production version
    # YOUR CODE HERE
    current_prod_versions = [
        v for v in client.search_model_versions(f"name='{registry_name}'")
        if v.current_stage == "Production"
    ]


    # Step 2: Archive current production
    for v in current_prod_versions:
        # Transition current production to Archived
        client.transition_model_version_stage(registry_name, v.version, "Archived")

        # Log the archival event
        log_audit_event(
            registry_name,
            v.version,
            "Production",
            "Archived",
            actor,
            f"ROLLBACK: replaced by v{rollback_to_version}. {reason}"
    )

    # Step 3: Promote rollback target back to Production
    # YOUR CODE HERE
    client.transition_model_version_stage(registry_name, rollback_to_version, "Production")

    # Log the promotion
    log_audit_event(
        registry_name, rollback_to_version, "Archived", "Production",
        actor, f"ROLLBACK TARGET: {reason}"
    )

    print(f" Rolled back to version {rollback_to_version}")


# Execute rollback to version 1
v1_version = client.get_latest_versions(REGISTRY_NAME, stages=["Archived"])
if v1_version:
    rollback_production(
        registry_name       = REGISTRY_NAME,
        rollback_to_version = v1_version[0].version,
        reason              = "Production F1 dropped from 0.72 to 0.41 after v2 deployment",
        actor               = "ml-lead@loanGuard.ai"
    )

print()
print("Final Audit Log:")
print("=" * 110)
print_audit_log()

Simulating a bad deployment...
Bad model (version 3) promoted to Production - F1: 0.6973
  Production is now degraded!

──────────────────────────────────────────────────
Initiating rollback...
──────────────────────────────────────────────────
 Rolled back to version 2

Final Audit Log:
Timestamp                  Model                  v    From         To           Actor              Reason
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
2026-03-13T10:33:55        LoanDefaultDetector    1    None         Staging      ci-system          Automated: offline eval passed F1 threshold
2026-03-13T10:33:55        LoanDefaultDetector    1    Staging      Production   ml-lead@loanGuard.ai Manual approval: A/B test shows +3% approval lift
2026-03-13T10:33:59        LoanDefaultDetector    3    Staging      Production   auto-deploy-pipeline Automated promotion - metric check bypassed (BUG)
2026-03-13T10:33:59        LoanDefaultDetecto

Registered model 'LoanDefaultDetector' already exists. Creating a new version of this model...
Created version '3' of model 'LoanDefaultDetector'.


###  4D - Governance Query

Answer the governance questions that LoanGuard's new engineer asked - programmatically.

In [24]:
#  BEGINNER 4D - Answer governance questions programmatically
# ------------------------------------------------------------

print("╔══════════════════════════════════════════════════════╗")
print("║        LoanGuard Model Governance Report             ║")
print("╚══════════════════════════════════════════════════════╝")
print()

# Q1: What model is in production right now?
# YOUR CODE HERE - query the registry and print name + version + stage
prod_models = [v for v in client.search_model_versions(f"name='{REGISTRY_NAME}'")
               if v.current_stage == "Production"]
print(f"Q1 - Current production model:")
for m in prod_models:
    print(f"     {m.name}  version {m.version}  ({m.current_stage})")

print()

# Q2: When was it promoted to production?
# Hint: look in AUDIT_LOG for the most recent 'Production' event
# YOUR CODE HERE

prod_events = [e for e in AUDIT_LOG if e['to_stage'] == "Production"]
if prod_events:
    latest_prod_event = sorted(prod_events, key=lambda x: x['timestamp'], reverse=True)[0]
    print(f"Q2 - Promoted to production at: {latest_prod_event['timestamp']}")
else:
    print("Q2 - No production events found")

print()


# Q3: Who approved it?
# Hint: look for actor in the Production audit event
# YOUR CODE HERE

if prod_events:
    print(f"Q3 - Approved by: {latest_prod_event['actor']}")
else:
    print("Q3 - No actor found")

print()

# Q4: What data trained it?
# Hint: use MlflowClient to get the run that produced it, then check tags
# YOUR CODE HERE

if prod_models:
    run_id = prod_models[0].run_id
    run_data = client.get_run(run_id)
    dataset_tag = run_data.data.tags.get("dataset_version", "Unknown")
    print(f"Q4 - Training dataset: {dataset_tag}")
else:
    print("Q4 - No production model found")

print()

print()

# Q5: What were the production model's metrics?
# YOUR CODE HERE

if prod_models:
    metrics = run_data.data.metrics
    metrics_str = ", ".join([f"{k}={v:.4f}" for k, v in metrics.items()])
    print(f"Q5 - Metrics at registration: {metrics_str}")
else:
    print("Q5 - No production model found")

╔══════════════════════════════════════════════════════╗
║        LoanGuard Model Governance Report             ║
╚══════════════════════════════════════════════════════╝

Q1 - Current production model:
     LoanDefaultDetector  version 2  (Production)

Q2 - Promoted to production at: 2026-03-13T10:33:59.720951

Q3 - Approved by: ml-lead@loanGuard.ai

Q4 - Training dataset: v1


Q5 - Metrics at registration: accuracy=0.8475, precision=0.8623, recall=0.8875, f1=0.8747, auc=0.9296


In [25]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    assert len(AUDIT_LOG) >= 2, 'Audit log must have at least 2 events'
    passed.append('Audit log has entries')
except Exception as e:
    failed.append(('Audit log has entries', str(e)))

try:
    assert all('timestamp' in e for e in AUDIT_LOG), 'Each event needs a timestamp'
    passed.append('Audit log has timestamps')
except Exception as e:
    failed.append(('Audit log has timestamps', str(e)))

try:
    assert (LAB_DIR / 'audit_log.json').exists(), 'audit_log.json not written to disk'
    passed.append('Audit log persisted')
except Exception as e:
    failed.append(('Audit log persisted', str(e)))

try:
    assert len(client.get_latest_versions(REGISTRY_NAME, stages=['Production'])) > 0, 'No model in Production stage'
    passed.append('Production model exists')
except Exception as e:
    failed.append(('Production model exists', str(e)))

try:
    assert len([e for e in AUDIT_LOG if 'ROLLBACK' in e.get('reason','')]) > 0, 'No rollback event in audit log'
    passed.append('Rollback executed')
except Exception as e:
    failed.append(('Rollback executed', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed: print(f'  {p}')
if failed:
    print('\nFailed:')
    for f,e in failed: print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')


 Passed: 5/5
  Audit log has entries
  Audit log has timestamps
  Audit log persisted
  Production model exists
  Rollback executed

All checks passed!


---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 4

Build a complete governance system:

**Requirements:**
1. `transition_stage()` with stage validation (only allow valid stage progressions: None→Staging, Staging→Production, Production→Archived)
2. `log_audit_event()` that writes to both in-memory list and a JSON file on disk
3. `rollback_production()` - archives current prod, promotes target, logs both events
4. `GovernanceReport` class with methods:
   - `current_production()` - returns name, version, stage, promoter, promotion time
   - `full_changelog(model_name)` - returns complete event history from audit log
   - `models_in_stage(stage)` - returns all models currently in a given stage
5. Simulate the full LoanGuard lifecycle: register v1 → staging → production → register bad v2 → production → rollback → archive v2
6. Print the final governance report answering all 5 questions from 4D

**Bonus:** Add a gate that blocks promotion to Production if the model's F1 score (from MLflow metrics) is below 0.65

In [26]:
#  ADVANCED - Task 4 Full Implementation
# -----------------------------------------------------------------
# First, decide which stage changes are allowed: None → Staging → Production → Archived.
# Make a function transition_stage() that moves a model and checks if the move is valid.
# Every time a stage changes, call log_audit_event() to keep track of who did what and when.
# Store the audit events both in memory and in a JSON file, so we don’t lose history.
# Write rollback_production() to take the bad model out of Production and put the good one back.
# Make a GovernanceReport class so we can ask: “What’s in Production?” or “Show all events for this model.”
# Register the first model version v1, move it to Staging, then promote it to Production.
# Register a second, weaker model v2 and try to promote it, but block it if F1 is too low (<0.65).
# If v2 causes problems, rollback to v1 and archive v2, logging everything.
# Finally, print a neat governance report showing the current model, who approved it, when, what data trained it, and its metrics.


#  ADVANCED - Task 4 Full Implementation
# ---------------------------------------------------------

import json
from datetime import datetime
from pathlib import Path
import mlflow
from mlflow.tracking import MlflowClient

# Directory to store audit logs
LAB_DIR = Path("./lab")
LAB_DIR.mkdir(exist_ok=True)
AUDIT_LOG = []

# Registry name
REGISTRY_NAME = "LoanDefaultDetector"
client = MlflowClient()

# Valid stage transitions
VALID_TRANSITIONS = {
    "None": ["Staging"],
    "Staging": ["Production"],
    "Production": ["Archived"]
}

# ---------------------------------------------------------
# 1. Stage transition with validation and logging
# ---------------------------------------------------------
def transition_stage(name, version, new_stage, actor="system", reason="", f1_gate=None):
    current_stage = client.get_model_version(name, version).current_stage

    # Skip if already in target stage
    if current_stage == new_stage:
        print(f"{name} v{version} already in stage {new_stage}, skipping.")
        return

    if new_stage not in VALID_TRANSITIONS.get(current_stage, []):
        raise ValueError(f"Invalid stage transition: {current_stage} → {new_stage}")

    # F1 gate
    if f1_gate and new_stage == "Production":
        run_id = client.get_model_version(name, version).run_id
        f1 = client.get_run(run_id).data.metrics.get("f1", 0)
        if f1 < f1_gate:
            print(f"Blocked promotion: F1={f1:.4f} below threshold {f1_gate}")
            return

    client.transition_model_version_stage(name, version, new_stage,
                                          archive_existing_versions=(new_stage=="Production"))

    log_audit_event(name, version, current_stage, new_stage, actor, reason)

# ---------------------------------------------------------
# 2. Audit logging
# ---------------------------------------------------------
def log_audit_event(model_name, version, from_stage, to_stage, actor, reason):
    event = {
        "timestamp": datetime.now().isoformat(),
        "model_name": model_name,
        "version": version,
        "from_stage": from_stage,
        "to_stage": to_stage,
        "actor": actor,
        "reason": reason
    }
    AUDIT_LOG.append(event)

    # Save to JSON
    log_path = LAB_DIR / "audit_log.json"
    if log_path.exists():
        existing = json.loads(log_path.read_text())
    else:
        existing = []
    existing.append(event)
    log_path.write_text(json.dumps(existing, indent=2))

# ---------------------------------------------------------
# 3. Rollback production
# ---------------------------------------------------------
def rollback_production(name, rollback_version, actor, reason=""):
    # Find current Production model(s)
    current_prod = [v for v in client.search_model_versions(f"name='{name}'")
                    if v.current_stage == "Production"]
    # Archive current Production
    for v in current_prod:
        client.transition_model_version_stage(name, v.version, "Archived")
        log_audit_event(name, v.version, "Production", "Archived",
                        actor, f"ROLLBACK: replaced by v{rollback_version}. {reason}")

    # Promote rollback target
    client.transition_model_version_stage(name, rollback_version, "Production")
    log_audit_event(name, rollback_version, "Archived", "Production",
                    actor, f"ROLLBACK TARGET: {reason}")

# ---------------------------------------------------------
# 4. Governance report class
# ---------------------------------------------------------
class GovernanceReport:
    def __init__(self, audit_log):
        self.audit_log = audit_log

    def current_production(self):
        prod_events = [e for e in self.audit_log if e["to_stage"]=="Production"]
        if not prod_events: return None
        latest = sorted(prod_events, key=lambda x: x["timestamp"], reverse=True)[0]
        return latest

    def full_changelog(self, model_name):
        return [e for e in self.audit_log if e["model_name"]==model_name]

    def models_in_stage(self, stage):
        stages = {}
        for e in self.audit_log:
            stages[e["version"]] = e["to_stage"]
        return [v for v,s in stages.items() if s==stage]

# ---------------------------------------------------------
# 5. Simulate LoanGuard lifecycle
# ---------------------------------------------------------
# v1 registration → Staging → Production
v1_model = mlflow.register_model(f"runs:/{BEST_RUN_ID}/model", REGISTRY_NAME)
v1 = v1_model.version
transition_stage(REGISTRY_NAME, v1, "Staging", actor="ci-system",
                 reason="v1 passed offline eval F1>0.72")
transition_stage(REGISTRY_NAME, v1, "Production", actor="ml-lead@loanGuard.ai",
                 reason="v1 approved for Production", f1_gate=0.65)

# v2 weak model → attempt Production promotion → fails F1 gate
weak_model = LogisticRegression(C=0.001, max_iter=100, random_state=42)
with mlflow.start_run(run_name="WeakModel_bad_deploy") as bad_run:
    weak_model.fit(X_train, y_train)
    weak_preds = weak_model.predict(X_test)
    mlflow.log_metric("f1", f1_score(y_test, weak_preds))
    mlflow.sklearn.log_model(weak_model, "model")
    bad_run_id = bad_run.info.run_id
v2_model = mlflow.register_model(f"runs:/{bad_run_id}/model", REGISTRY_NAME)
v2 = v2_model.version
transition_stage(REGISTRY_NAME, v2, "Staging", actor="ci-system",
                 reason="v2 passed unit tests")
transition_stage(REGISTRY_NAME, v2, "Production", actor="ml-lead@loanGuard.ai",
                 reason="v2 attempted Production promotion", f1_gate=0.65)

# Rollback → v1
rollback_production(REGISTRY_NAME, v1, actor="ml-lead@loanGuard.ai",
                    reason="v2 F1 too low")

# Archive v2
transition_stage(REGISTRY_NAME, v2, "Archived", actor="ml-lead@loanGuard.ai",
                 reason="v2 archived after failed Production")

# ---------------------------------------------------------
# 6. Print governance report answering 4D questions
# ---------------------------------------------------------
report = GovernanceReport(AUDIT_LOG)
prod = report.current_production()
if prod:
    print(f"Current production model: {REGISTRY_NAME} v{prod['version']}")
    print(f"Promoted at: {prod['timestamp']}")
    print(f"Approved by: {prod['actor']}")
    run_data = client.get_run(client.get_model_version(REGISTRY_NAME, prod["version"]).run_id)
    dataset = run_data.data.tags.get("dataset_version","Unknown")
    metrics = ", ".join([f"{k}={v:.4f}" for k,v in run_data.data.metrics.items()])
    print(f"Training dataset: {dataset}")
    print(f"Metrics at registration: {metrics}")


Registered model 'LoanDefaultDetector' already exists. Creating a new version of this model...
Created version '4' of model 'LoanDefaultDetector'.


LoanDefaultDetector v5 already in stage Archived, skipping.
Current production model: LoanDefaultDetector v4
Promoted at: 2026-03-13T10:34:03.351824
Approved by: ml-lead@loanGuard.ai
Training dataset: v1
Metrics at registration: accuracy=0.8475, precision=0.8623, recall=0.8875, f1=0.8747, auc=0.9296


Registered model 'LoanDefaultDetector' already exists. Creating a new version of this model...
Created version '5' of model 'LoanDefaultDetector'.


---
#  Lab Complete - Reflection
---

##  What You Built

| Task | What you implemented | Real-world equivalent |
|------|----------------------|----------------------|
| **Task 1** | DVC pointer files, drift detection, schema validator | DVC + Great Expectations |
| **Task 2** | MLflow experiment runs, multi-model comparison, artifact retrieval | MLflow Tracking Server |
| **Task 3** | Versioned model artifacts, semantic version logic, model registration | MLflow Registry + release process |
| **Task 4** | Stage transitions, audit log, rollback, governance queries | MLflow Registry Lifecycle + compliance |

---

## The Full Stack - How It Connects

```
loans_v1.csv  →  validate_schema()  →  [PASSES]  →  Training Run
                                                         │
                                              mlflow.log_params()
                                              mlflow.log_metrics()
                                              mlflow.log_model()
                                                         │
                                              register_model("LoanDefaultDetector")
                                                         │
                                         None → Staging → Production
                                                    (with audit log)
                                                         │
                             If it fails →  rollback_production()  →  Archived
```

---

##  Reflection Questions

Answer these in the cell below before the session ends:

1. Which task felt most like something you'd use immediately in your current project?
2. Where in LoanGuard's original problem could a schema validator have saved the most time?
3. If you had to pick ONE layer of the versioning stack to implement tomorrow - which would it be and why?

In [27]:
#  Your Reflection Answers
# ----------------------------------------------------

reflection = {
    "most_immediately_useful": """
    Task 4 felt most immediately useful because implementing stage transitions,
    audit logs, and rollback is directly applicable to any production ML system.
    Being able to see what’s in production and safely revert models is critical.
    """,
    "where_schema_validation_would_have_helped": """
    A schema validator would have saved the most time at the very beginning,
    when ingesting new loan data. Detecting missing or mis-typed columns before
    training would prevent wasted runs and misleading metrics.
    """,
    "first_layer_to_implement": """
    If I had to pick one layer to implement tomorrow, it would be the model
    versioning and registry lifecycle (Task 3 + 4). This ensures that every
    model is tracked, auditable, and safe to deploy or rollback.
    """
}

for q, a in reflection.items():
    print(f"Q: {q}")
    print(f"A: {a.strip()}")
    print()



Q: most_immediately_useful
A: Task 4 felt most immediately useful because implementing stage transitions,
    audit logs, and rollback is directly applicable to any production ML system.
    Being able to see what’s in production and safely revert models is critical.

Q: where_schema_validation_would_have_helped
A: A schema validator would have saved the most time at the very beginning,
    when ingesting new loan data. Detecting missing or mis-typed columns before
    training would prevent wasted runs and misleading metrics.

Q: first_layer_to_implement
A: If I had to pick one layer to implement tomorrow, it would be the model
    versioning and registry lifecycle (Task 3 + 4). This ensures that every
    model is tracked, auditable, and safe to deploy or rollback.

